In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)


In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)


In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [6]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )


In [7]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [8]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [9]:
print(response.output_text)

The provided context does not include specific instructions on how to create a dashboard in Evidently. Please provide more details or check the official documentation for guidance on this process.


Making It Agentic

In [10]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


In [11]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [12]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [13]:
tool_call = response.output[0]
tool_call


ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_VkKV0DeqdRCWuSEgKvGoVO6x', name='search', type='function_call', id='fc_044d3379836a8673006991bafffb988197864271d2c3ff161e', status='completed')

In [14]:
messages.append(tool_call)

In [15]:
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [16]:
arguments = json.loads(tool_call.arguments)
arguments


{'query': 'create dashboard in Evidently'}

In [17]:
search_results = search(query='create dashboard in Evidently')

In [18]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': 'Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\n\n<Check>\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\n</Check>\n\n## Adding Tabs\n\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\n\n**To add a Tab**:\n\n- Enter "Edit" mode on the Dashboard (top right corner).\n- Click the plus sign with "add Tab" on the left.\n- To create a custom Tab, select "empty" and enter a name.\n\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\n\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\n\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, the Panels will appear empty un

In [19]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}


In [20]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_VkKV0DeqdRCWuSEgKvGoVO6x', name='search', type='function_call', id='fc_044d3379836a8673006991bafffb988197864271d2c3ff161e', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_VkKV0DeqdRCWuSEgKvGoVO6x',
  'output': '[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\\n## Adding Tabs

In [21]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

4075

In [22]:
print(response.output_text)


To create a dashboard in Evidently, follow these steps:

### 1. Create a Project
Before you can create a dashboard, make sure you have an Evidently Project, as you'll need reports with evaluation results added to it.

### 2. Adding Tabs
- **Enter Edit Mode**: Click on "Edit" in the top right corner of your dashboard.
- **Add a Tab**: Click the plus sign with “add Tab” on the left. You can create a custom Tab by selecting "empty" and entering a name.
- **Note**: You can also start with pre-built Tabs which come with preset Panel combinations.

### 3. Adding Panels
You can add various types of Panels (text, counters, pie charts, line plots, and bar plots) to visualize your evaluation results:
- **To add a Panel**:
  - Stay in "Edit" mode.
  - Click on the "Add Panel" button.
  - Follow the prompts to configure the panel.
  - Select the Tab where you want to add the Panel and click "Save".

### 4. Configure Your Panel
- Select Metrics corresponding to the data available in your Reports.
-

Structured Output

In [23]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [24]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

4299

In [25]:
4299 - 4075

224

In [26]:
rag_response = response.output_parsed
print(rag_response.answer)

Creating a dashboard in Evidently involves several steps, including adding tabs and panels to visualize evaluation results. Here’s a simple guide to get you started:

### Step 1: Access the Dashboard Editor
1. Navigate to the Dashboard section of your project.
2. Click on the "Edit" mode located at the top right corner of the Dashboard interface.

### Step 2: Add Tabs (Optional)
- **To add a Tab**:
  - Click the plus sign to "Add Tab" on the left.
  - Choose "empty" to create a new custom Tab and enter a name.
  - You can also choose pre-built Tabs if related Metrics are available.

### Step 3: Add Panels
1. Click on the "Add Panel" button in Edit mode.
2. Follow the prompts to configure your panel:
   - Select the metrics to visualize (ensure they match the names logged in your Reports).
   - You can choose from various panel types like counters, pie charts, line plots, and more.
   - Configure aggregation options (sum, average, last value, etc.).
3. Preview your setup and click "Save